In [ ]:
!dd if=/dev/urandom of=sample_file.bin bs=128 count=1
!openssl sha256 sample_file.bin

In [ ]:
import mmap

import ctypes
import json
import hashlib
import numpy as np
import random
import time

with open('sample_file.bin', 'r+b') as f:
    with mmap.mmap(f.fileno(), length=0, access=mmap.ACCESS_WRITE) as mm:
        # Memory mapped file mm behaves like a bytearray
        sha256_hash = hashlib.sha256(mm).hexdigest()
        print(f'Hash before sort: {sha256_hash}')

        # Treat memory mapped file bytes as integers array in memory.
        # Sorting them.
        int_array = np.frombuffer(mm, dtype=np.uint32)
        print('Bytes as array of garbage 32 unsigned integers')
        print(int_array[:32]) # Unsorted
        int_array.sort()
        print('Bytes as array of sorted 32 unsigned integers')
        print(int_array[:32]) # Sorted
        mm.flush()
        
        # numpy array must be closed before memory mapped file.
        # BufferError: cannot close exported pointers exist
        del int_array

        sha256_hash = hashlib.sha256(mm).hexdigest()
        print(f'Hash after sort: {sha256_hash}')

In [ ]:
with open('sample_file.bin', 'rb') as f:
    file_content = f.read()
    sha256_hash = hashlib.sha256(file_content).hexdigest()
    print(f'Checking that file content changed indeed: {sha256_hash}')
    
    with mmap.mmap(f.fileno(), length=0, access=mmap.ACCESS_READ) as mm:
        sha256_hash = hashlib.sha256(mm).hexdigest()
        print(f'Hash computed from memory mapped file: {sha256_hash}')

        try:
            mm[0] = b'x'
            assert('Cannot write to read-only mmap')
        except TypeError as e:
            #TypeError: mmap can't modify a readonly memory map.
            print('Exception as expected: ', e)

In [ ]:
with open('sample_file.bin', 'r+b') as f:
    with mmap.mmap(f.fileno(), length=0, access=mmap.ACCESS_WRITE) as mm:
        # Memory mapped file mm behaves like a bytearray
        sha256_hash = hashlib.sha256(mm).hexdigest()
        print(sha256_hash)

        # Treating buffer as signed integers (~ half of them will be negative)
        IntArrayType = ctypes.c_int * (len(mm) // ctypes.sizeof(ctypes.c_int))
        int_array = IntArrayType.from_buffer(mm)
        
        j = json.dumps({'numbers' : [n for n in int_array]}, indent=4).encode('utf-8')
        print('Previously sorted 32 bit unsigned integers treated as signed')
        print(j)

        # Cannot resize memory mapped file without closing integers arrays still holding a reference.
        # BufferError: cannot close exported pointers exist
        del int_array

        # Deceptively called function "truncate" works just fine to grow the file size forward. 
        f.truncate(len(j))

        
        mm.resize(len(j))
        mm[0:] = j
        mm.flush()

In [ ]:
with open('sample_file.bin', 'r') as f:
    j = json.load(f)
    print('Reading presumably json file back')
    print(j)